**Bigger-font variant (2026-08 supervisor feedback item 2).** Copied from the original simple-plot notebook, not edited in place — same convention as `100_versions_pie_plot_simple_bigfont.ipynb` and the original-vs-simple-plot split before it. Reads from the bigfont chart source (`100_pie_charts_simple_bigfont/`) and writes to a separate `*_bigfont` post/output tree throughout, so nothing here collides with the existing simple-plot pilot data or its results. See `docs/SESSION_HANDOFF.md` for context.

---
# Benchmarking Qwen3 8B Vision LLM (Simple Plot)


In [ ]:
# This installs torch 2.6.0 and torchvision 0.21.0 pinned to cu124 because the
# GPU server's NVIDIA driver supports CUDA 12.4 but not CUDA 13.0. torchaudio is
# explicitly uninstalled and excluded because it is not needed for Qwen3-VL and
# its cu124 build conflicts with the server's CUDA 13 runtime libraries, causing
# an OSError on import. transformers is installed from source because Qwen3-VL
# support is not yet available in a stable PyPI release.

import sys, subprocess

# Uninstall torchaudio if it exists from a previous session
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

# Install torch stack (no torchaudio)
subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

# Install transformers + dependencies
subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate", "qwen-vl-utils",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

In [ ]:
!nvidia-smi

# Loading most recent version of Vision Qwen to benchmark


In [ ]:
import sys
sys.path.append("/home/jovyan")

from config_hf_token import HF_TOKEN
from huggingface_hub import login

login(token=HF_TOKEN)

In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("✅ Loaded successfully")

In [ ]:
MODEL_NAME = "qwen3-vl-8b" 

In [ ]:
import sys
from pathlib import Path

# Reuse the same benchmarking utils (quali_benchmarking.py, quanti_benchmarking_*.py)
# from the original benchmarking/ notebook instead of duplicating them — only the
# image source paths differ for the simple-plot pie charts.
ROOT_DIR = Path().resolve().parent
sys.path.insert(0, str(ROOT_DIR / "benchmarking"))

## QUALITATIVE
#### Baseline - gender neutral- news outlets - correct/incorrect - visible 0s

In [ ]:
import sys
from pathlib import Path
#sys.path.append(str(Path().resolve().parent))
from utils.quali_benchmarking import describe_social_media_post_qwen, save_output_txt, output_exists

In [ ]:
# --- Configuration ---
BASE_DIR = Path().resolve()
PROMPT_NAME = "simple"

folders_to_process = ["correct", "incorrect"]
SIMPLE_PLOT_DIR = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont"

# --- Processing Loop ---
# Same 4-profile, image-001 pilot set as the original benchmarking notebook
# (gender-neutral "remy_ashford" + 3 news outlets), just sourced from the
# simple-plot chart images instead of the original ones.
for folder in folders_to_process:
    suffix = "c" if folder == "correct" else "i"
    candidates = [
        SIMPLE_PLOT_DIR / folder / "PNGs" / f"001_remy_ashford_{suffix}.png",
        SIMPLE_PLOT_DIR / folder / "PNGs" / "news" / f"001_fox_news_{suffix}.png",
        SIMPLE_PLOT_DIR / folder / "PNGs" / "news" / f"001_ny_times_{suffix}.png",
        SIMPLE_PLOT_DIR / folder / "PNGs" / "news" / f"001_reuters_{suffix}.png",
    ]
    image_files = [p for p in candidates if p.exists()]
    missing = [p for p in candidates if not p.exists()]
    if missing:
        print(f"⚠ Missing files for [{folder}]: {missing}")

    print(f"\nProcessing {len(image_files)} image(s) from folder: [{folder.upper()}]")
    for img_path in image_files:
        str_image_path = str(img_path)

        if output_exists(str_image_path, folder, BASE_DIR, MODEL_NAME, PROMPT_NAME):
            print(f"⏭ Skipping: {img_path.name}")
            continue

        result = describe_social_media_post_qwen(str_image_path, model, processor, device)
        save_output_txt(str_image_path, result, folder_name=folder, base_dir=BASE_DIR, model_name=MODEL_NAME, prompt_name=PROMPT_NAME)

print("\nAll folders processed successfully!")

### Analysis

#### 1. Fox News:

#### 2. The New York Times

#### 3. Remy Ashford

#### 4. Reuters


## QUANTITATIVE

**Tests 1-3** establish a baseli: — how well can the model read charts and verify claims when there are no social signals present.

**Test 4** introduces social signals (reaction metrics) and : s — does the model's claim verification accuracy change when the post appears highly liked, or highly reacted to with angry/sad emotiesis scope?

## 1) Extensive analysis of all baseline examples (1 gender neutral user - 3 news outlets) 

In [ ]:
import sys
from pathlib import Path
from utils.quanti_benchmarking_1_details import (
    PROMPT_VERSIONS, TWO_CALL_PROMPT_VERSIONS,
    benchmark_image, benchmark_image_two_call, output_exists_json,
    ask_question
)

# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-1-gn-news-extensive"
ASK_FN = ask_question
variants = ["correct", "incorrect"]
SIMPLE_PLOT_DIR = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont"

# --- Build image list (same 4-profile, image-001 pilot set as the qualitative test) ---
all_images = []
for variant_folder in variants:
    suffix = "c" if variant_folder == "correct" else "i"
    candidates = [
        SIMPLE_PLOT_DIR / variant_folder / "PNGs" / f"001_remy_ashford_{suffix}.png",
        SIMPLE_PLOT_DIR / variant_folder / "PNGs" / "news" / f"001_fox_news_{suffix}.png",
        SIMPLE_PLOT_DIR / variant_folder / "PNGs" / "news" / f"001_ny_times_{suffix}.png",
        SIMPLE_PLOT_DIR / variant_folder / "PNGs" / "news" / f"001_reuters_{suffix}.png",
    ]
    for png in candidates:
        if png.exists():
            all_images.append((png.stem, str(png)))

# --- Run single-call prompt versions ---
for prompt_version, prompt_text in PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning prompt version: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image(image_path, image_name, prompt_version, prompt_text, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)
'''
# --- Run two-call prompt versions ---
for prompt_version, prompt_template in TWO_CALL_PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning two-call prompt version: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image_two_call(image_path, image_name, prompt_version, prompt_template, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)
'''
print("\n✅ All versions complete.")

### Accuracy calculations for each version

In [ ]:
from utils.quanti_benchmarking_1_analysis import run_accuracy_analysis

BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-1-gn-news-extensive"  # same as in benchmarking notebook
run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)

## Prior to adding the two call prompts (v1-v6 only)
- v1–v2: Too vague — "accurately describe" invites hallucinated criteria
- v3: Forces the model to reason in a specific direction, but the conditional phrasing ("if… reply correct") is hard for VLMs to follow reliably
- v4–v5: Better, but "find the percentage for Pop and Latin" may cause the model to report numbers correctly but then flip its verdict
- v6: The step-by-step is the best approach but still ends ambiguously

**The core issue here might be missing explicit numeric grounding**
- None of the prompts ask the model to state the numbers before giving a verdict. VLMs tend to shortcut directly to a label when prompted for a binary answer.
- The model likely reads 23.5% and 11.0% correctly (proven by the chart_percentages questions succeeding), but when asked for a verdict in one shot, it could be regressing to a bias.

## Next solution approach: two call prompts (v7-v9)

Instead of asking the model one question like "does the text match the chart?" and hoping it figures everything out on its own, the two-call approach breaks it into two steps:

1. I ask the model to read the specific numbers from the chart (Pop % and Latin %)
2. I give it those numbers directly and ask if the claim matches

The idea is that the model was likely reading the chart correctly, but stumbling when asked to simultaneously read, reason, and give a verdict in one go. By separating perception from reasoning, we give it a better chance of getting the final answer right.

**28/05 Giordano comment**: We will not be conducting the experiment with two call prompts, chances are the bigger the model the better it performs at determining whether the claim is correct or not.

## 2) Focusing on gender neutral user - using best performing prompt - only asking wether claim is correct or not - 100 versions of visualization remy ashford - 50/50 correct incorrect ratio - store the order provided to the LLM
After part 1, should definitely be using the two call approach here.


In [ ]:
# Not needed for simple-plot data — images are already unzipped under spotify_pie_plot/pie_plot_posts/


In [ ]:
import sys
import random
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.quanti_benchmarking_2_claim_only import (
    PROMPT_VERSIONS, TWO_CALL_PROMPT_VERSIONS,
    benchmark_image, benchmark_image_two_call, output_exists_json,
    ask_question
)

# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-2-gn-claim-only"
ASK_FN = ask_question
SEED = 42
SAMPLE_SIZE = 50

# --- Build paired sample ---
correct_dir = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/correct/PNGs"
incorrect_dir = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/incorrect/PNGs"
all_numbers = sorted([p.name.split("_")[0] for p in correct_dir.glob("*_remy_ashford_c.png")])
print(f"Found {len(all_numbers)} images in {correct_dir}")

random.seed(SEED)
selected_numbers = sorted(random.sample(all_numbers, SAMPLE_SIZE))
all_images = []
for num in selected_numbers:
    all_images.append((f"{num}_correct", str(correct_dir / f"{num}_remy_ashford_c.png")))
    all_images.append((f"{num}_incorrect", str(incorrect_dir / f"{num}_remy_ashford_i.png")))
print(f"Selected {len(selected_numbers)} pairs → {len(all_images)} images total")

# --- Run one-call versions ---
for prompt_version, prompt_text in PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image(image_path, image_name, prompt_version, prompt_text, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)

'''
# --- Run two-call versions ---
for prompt_version, prompt_template in TWO_CALL_PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image_two_call(image_path, image_name, prompt_version, prompt_template, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)
'''

print("\n✅ All remy-ashford versions complete.")

In [ ]:
from utils.quanti_benchmarking_2_analysis import run_accuracy_analysis

BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-2-gn-claim-only"
run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)




## 3) Ask about nr in pie charts - both genres - 100 - 50/50
- Reusing all_images from prevous benchmark, in other words, using the same 50 pairs used for previous benchmark

**Why**
- Direct comparability: if test 2 (claim verification) and test 3 (percentage reading) use the same images, we can link results. For example: "the model read the percentages correctly on image 042 but still got the claim wrong": that's a meaningful finding about where reasoning breaks down.
- Controls for image variability: if the sets differ, a performance difference between tests could be due to one set happening to have easier images rather than the task itself being easier.
- Cleaner narrative 


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.quanti_benchmarking_3_percentages import (
    benchmark_image_percentages, output_exists_json, ask_question
)

# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-3-gn-percentages"
ASK_FN = ask_question

# --- Run ---
print(f"\n{'='*60}")
print("Running test-3: percentage extraction")
print(f"{'='*60}")
for image_name, image_path in all_images:
    if output_exists_json(image_name, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
        print(f"⏭ Skipping: {image_name}")
        continue
    benchmark_image_percentages(image_path, image_name, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)
print("\n✅ Test 3 complete.")

In [ ]:
from utils.quanti_benchmarking_3_analysis import run_accuracy_analysis
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-3-gn-percentages"
run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)

## old 4 Nr of reactions - cover all reactions - equally distributed (will probably keep out because unrealistic)
Does the overall volume of engagement metrics influence the model's claim verification accuracy, independent of reaction type distribution?

By keeping all reactions equal we are controlling for reaction type bias because the model can't be swayed by seeing mostly angry vs mostly love reactions. 

In [ ]:
# Not needed for simple-plot data — images are already unzipped under spotify_pie_plot/pie_plot_posts/


The same 50 numbers are reused across all 6 scale values, giving 600 images total per prompt version (50 pairs × 6 scale values × 2 variants)

# 4) Nr of reactions - more realistic -  X total across all reaction types (likes + loves + hahas etc. combined), x being the log numebr so 10, 100, 1000, etc.
Posts were generated under two reaction conditions: uniform, in which all reaction types were set equally, and realistic, in which reactions were distributed using log-scaled weights to approximate empirical engagement patterns on social media.

With uniform reactions, every post at scale_value=1000 showed exactly 1000 likes, 1000 loves, 1000 hahas etc. — which is something that essentially never occurs on real Facebook and could itself be a signal to the VLM that something artificial is happening. The realistic condition removes that artificiality while keeping scale_value as a clean, interpretable independent variable representing **total engagement volume**.

**Why Jitter Matters**
Without jitter, every image index at a given scale_value would produce identical reaction counts. For example, at scale_value=1000 every single one of your 100 images would show:


- Emoji order is always like, love, haha, wow, sad, angry — fixed by the change in the .py file
- Values vary across images because shares are shuffled using seed=i — so image 001 always gets the same distribution but different from image 002
- Total reactions always sum to approximately scale_value — Interpretation 1
- Files land in realistic/ keeping them separate from your uniform/ condition
- Reproducible — rerunning will generate identical files

In [ ]:
# Not needed for simple-plot data — images are already unzipped under spotify_pie_plot/pie_plot_posts/


In [ ]:
import sys
import random
from pathlib import Path

sys.path.append(str(Path().resolve().parent))
from utils.quanti_benchmarking_4_reactions import (
    PROMPT_VERSIONS, benchmark_image, output_exists_json, ask_question
)

# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-4-metrics-realistic-claim-only"
MODEL_NAME = "qwen3-vl-8b"
ASK_FN = ask_question
SEED = 42
SAMPLE_SIZE = 50
REACTION_VALUES = [10, 100, 1000, 10000, 100000, 1000000]

# --- Build paired sample ---
correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/realistic"
incorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/realistic"

sample_dir = correct_base / "10"
all_numbers = sorted([p.name.split("_")[0] for p in sample_dir.glob("*_remy_ashford_c.png")])
print(f"Found {len(all_numbers)} images in {sample_dir}")

random.seed(SEED)
selected_numbers = sorted(random.sample(all_numbers, SAMPLE_SIZE))
print(f"Selected {len(selected_numbers)} numbers: {selected_numbers}")

all_images = []
for scale_value in REACTION_VALUES:
    for num in selected_numbers:
        all_images.append((
            f"{num}_correct_{scale_value}",
            str(correct_base / str(scale_value) / f"{num}_remy_ashford_c.png")
        ))
        all_images.append((
            f"{num}_incorrect_{scale_value}",
            str(incorrect_base / str(scale_value) / f"{num}_remy_ashford_i.png")
        ))

print(f"Total images to process: {len(all_images)}")

# --- Run ---
for prompt_version, prompt_text in PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image(image_path, image_name, prompt_version, prompt_text, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)

print("\n✅ All metrics versions complete.")

In [ ]:
from utils.quanti_benchmarking_4_analysis import run_accuracy_analysis

BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-4-metrics-realistic-claim-only"

run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)